# Train the filament U-Net (ImageNet-pretrained encoder) on Google Colab (free GPU)
**Before running:** upload these two files into a folder **`filseg`** in your Google Drive (My Drive → filseg):
* `filseg-code-gpu.zip` (from Documents) — upload only this code zip, not the older `filseg-code.zip`
* `filament-segmentation-2026 (1).zip` (from Downloads, ~700 MB)

Then: **Runtime → Change runtime type → T4 GPU → Save**, and **Runtime → Run all**.
Results are saved back to **My Drive → filseg → results**, so nothing is lost if Colab disconnects.

In [ ]:
# 1. Connect Google Drive (a pop-up asks you to allow access)
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil, torch
DRIVE = '/content/drive/MyDrive/filseg'
print(os.listdir(DRIVE))
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set Runtime > Change runtime type > T4 GPU')

In [ ]:
# 2. Unpack code and data onto Colab's fast local disk (~2-4 min)
CODE_ZIP = glob.glob(f'{DRIVE}/filseg-code-gpu*.zip')[0]
DATA_ZIP = glob.glob(f'{DRIVE}/filament-segmentation-2026*.zip')[0]
os.makedirs('/content/filseg', exist_ok=True)
!unzip -q -o "{CODE_ZIP}" -d /content/filseg
!unzip -q -o "{DATA_ZIP}" -d /content/filseg/data
os.chdir('/content/filseg')
os.makedirs('weights', exist_ok=True)
!pip -q install pycocotools segmentation-models-pytorch==0.5.0
!ls data/MAGFiLO_1.0_Kaggle_2026/train/train_images | wc -l

In [ ]:
MINUTES = 150   # training time budget (free Colab sessions last a few hours)
OUT = f'{DRIVE}/results'; os.makedirs(OUT, exist_ok=True)

## 3. Train (pre-processing cache first ~15-20 min, then training; progress printed every epoch)

In [ ]:
MODEL = "--arch smp --encoder resnet34 --crop 512 --batch 16 --lr 5e-4"   # ImageNet-pretrained ResNet-34 encoder
!python scripts/run.py train --data /content {MODEL} --minutes {MINUTES} 2>&1 | tee -a "{OUT}/train_log.txt"
shutil.copy('weights/unet.pt', OUT)   # save the model to Drive straight away

## 4. Tune threshold / fragment linking for PQ on the validation split

In [ ]:
!python scripts/run.py tune --method unet --n-val 60 2>&1 | tee "{OUT}/tune_log.txt" | tail -3

## 5. Submission file

In [ ]:
!python scripts/run.py submit --method unet --out submission_unet.csv
for f in glob.glob('weights/*') + ['submission_unet.csv']:
    shutil.copy(f, OUT)
print(sorted(os.listdir(OUT)))
print('Done. Download everything in My Drive > filseg > results')